## **Match Simulation Setup**
- simulate_match
- group_simulate
- get_round_of_32
- knockout_simulate
- wc_simulate
- monte_carlo_simulate

In [301]:
import numpy as np

def simulate_match(
    match_row, # this should contain date, home_team, away_team
    
    # =========================
    # MODELS (grouped)
    # =========================
    models,

    # models = {
    #     "goal_home": ...,
    #     "goal_away": ...,
    #     "yellow_home": ...,
    #     "yellow_away": ...,
    #     "red_home": ...,
    #     "red_away": ...,
    #     "corner_home": ...,
    #     "corner_away": ...
    # }
    
    # =========================
    # DATA
    # =========================
    elo_ratings, # elo ratings
    team_hist, # team history last 5 - 10 matches
    
    # =========================
    # FEATURE FUNCTIONS
    # =========================
    feature_engine,

    #feature_engine = {
    #    "combine_elo": combine_teams_elo_fn,
    #    "home_adv": is_home_advantage_fn,
    #    "match_features": add_match_features_fn
    #}

    # =========================
    # SETTINGS
    # =========================
    knockout=False,
    et_total=0.8
    ):

    """
    Simulate a football match using trained models to get probability of goals, cards, corners,
    then use poisson from numpy to get the exact values for these data

    In knockout mode, 
        - Average goals extra time in the WC ~0.8 goals per match, compute the expected goals
        - If draw after extra time, penalty will be True

    Returns:
        if knockout is False, return values will be a dict
        {  
            goals_home, goals_away, 
            yellow_home, yellow_away,
            red_home, red_away,
            corner_home, corner_away,
            result_str
        } 
        else, return values will be a dict
        {   
            goals_home, goals_away, 
            yellow_home, yellow_away,
            red_home, red_away,
            corner_home, corner_away,
            isPenalty,
            result_str
        }
        *result_str: "W" (A wins), "L" (B wins), "D" (draw)
    """
    
    host_countries = ["United States", "USA", "Canada", "Mexico"]
    goal_corner_cols = ["home_elo", "away_elo", "elo_diff", "home_advantage", "home_attack_rate", "home_defense_rate", "away_attack_rate", "away_defense_rate"]
    card_cols = goal_corner_cols + ["home_disc", "away_disc", "disc_diff", "disc_sum"]

    # =========================
    # FEATURE ENGINEERING
    # =========================
    match_row = feature_engine["combine_elo"](match_row, elo_ratings) # "home_elo", "away_elo", "elo_diff"
    
    match_row["home_advantage"] = int(
        match_row["home_team"].isin(host_countries)
    ) if hasattr(match_row["home_team"], "isin") else int(
        feature_engine["home_adv"](match_row["home_team"], host_countries)
    )
    
    print(match_row)
    print("reach")
    match_row = feature_engine["match_features"](match_row, team_hist) # "home_attack_rate", "home_defense_rate", "away_attack_rate", "away_defense_rate", "home_disc", "away_disc", "disc_diff", "disc_sum"
    print("reach111")
    # Reshape
    X_goals = match_row[goal_corner_cols].values.reshape(1, -1)
    X_cards = match_row[card_cols].values.reshape(1, -1)


    # =========================
    # PREDICT LAMBDAS
    # =========================
    lam_home_goal = models["goal_home"].predict(X_goals)[0]
    lam_away_goal = models["goal_away"].predict(X_goals)[0]

    lam_home_yellow = models["yellow_home"].predict(X_cards)[0]
    lam_away_yellow = models["yellow_away"].predict(X_cards)[0]

    lam_home_red = models["red_home"].predict(X_cards)[0]
    lam_away_red = models["red_away"].predict(X_cards)[0]

    lam_home_corner = models["corner_home"].predict(X_goals)[0]
    lam_away_corner = models["corner_away"].predict(X_goals)[0]


    # =========================
    # 90 MIN SIMULATION
    # =========================
    home_goals = np.random.poisson(max(0, lam_home_goal))
    away_goals = np.random.poisson(max(0, lam_away_goal))

    home_yellow = np.random.poisson(max(0, lam_home_yellow))
    away_yellow = np.random.poisson(max(0, lam_away_yellow))

    home_corners = np.random.poisson(max(0, lam_home_corner))
    away_corners = np.random.poisson(max(0, lam_away_corner))

    home_red = int(np.random.random() < max(0, lam_home_red))
    away_red = int(np.random.random() < max(0, lam_away_red))

    
    # =========================
    # EXTRA TIME (KO ONLY)
    # =========================
    isPenalty = False
    penalty_winner = None

    if knockout and home_goals == away_goals:

        total = lam_home_goal + lam_away_goal
        if total <= 0:
            h_share = 0.5
            a_share = 0.5
        else:
            h_share = lam_home_goal / total
            a_share = lam_away_goal / total

        lam_home_et = et_total * h_share
        lam_away_et = et_total * a_share

        et_home = np.random.poisson(lam_home_et)
        et_away = np.random.poisson(lam_away_et)

        home_goals += et_home
        away_goals += et_away

        penalty_winner = None

        if et_home == et_away:
            isPenalty = True
            if np.random.random() < 0.5:
                penalty_winner = match_row["home_team"]
            else:
                penalty_winner = match_row["away_team"]

    # =========================
    # RESULT
    # =========================
    if home_goals > away_goals:
        result_str = "W"
    elif home_goals < away_goals:
        result_str = "L"
    else:
        result_str = "D"

    if isPenalty:
        if penalty_winner == match_row["home_team"]:
            result_str = "W"
        elif penalty_winner == match_row["away_team"]:
            result_str = "L"

    result = {
        "home_goals": home_goals,
        "away_goals": away_goals,
        "home_yellow": home_yellow,
        "away_yellow": away_yellow,
        "home_red": home_red,
        "away_red": away_red,
        "home_corners": home_corners,
        "away_corners": away_corners,
        "penalty": isPenalty,
        "result_str": result_str
    }

    return result

In [302]:
import pandas as pd
def simulate_group_stage(stage_table_df, simulate_match_fn, models, elo_ratings, team_hist, feature_engine, fill_competition_stage_table = False, verbose = True):
    """
    Simulate group stage from a match schedule dataframe.
    """

    # =========================
    # PREP DATA
    # =========================
    stage_table_df = stage_table_df.copy()
    stage_table_df["date"] = pd.to_datetime(stage_table_df["date_utc"], utc=True)

    groups = stage_table_df["group"].unique()

    # =========================
    # PRECOMPUTE GROUP DATA
    # =========================
    group_matches_map = {
        g: stage_table_df[stage_table_df["group"] == g]
        for g in groups
    }

    group_teams_map = {
        g: pd.unique(df[["home_team", "away_team"]].values.ravel())
        for g, df in group_matches_map.items()
    }

    # =========================
    # INIT STATS
    # =========================
    group_stats = {}

    for g in groups:
        group_stats[g] = {
            t: {
                "pts": 0,
                "gf": 0,
                "ga": 0,
                "gd": 0,
                "w": 0,
                "d": 0,
                "l": 0,
                "yc": 0,
                "rc": 0,
                "corners": 0
            }
            for t in group_teams_map[g]
        }

    
    # store results for replay / debugging
    results = []

    # =========================
    # SIMULATE EACH MATCH
    # =========================
    for row in stage_table_df.itertuples(index=False):

        group = row.group
        stats = group_stats[group]
        group_teams = group_teams_map[group] 

        match_row = pd.DataFrame({
            "home_team": [row.home_team],
            "away_team": [row.away_team],
            "date": [row.date]
        })

        print("reach")
        result = simulate_match_fn(match_row, models, elo_ratings, team_hist, feature_engine)
        print("reach")
        
        home = row.home_team
        away = row.away_team

        home_goals = result["home_goals"]
        away_goals = result["away_goals"]
        home_yellow = result["home_yellow"]
        away_yellow = result["away_yellow"]
        home_red = result["home_red"]
        away_red = result["away_red"]
        home_corners = result["home_corners"]
        away_corners = result["away_corners"]
        result_str = result["result_str"]

        # =========================
        # FILL COMPETITION STAGE TABLE
        # =========================
        #predicted_home_goals	predicted_away_goals	corners	yellow_cards	red_cards	winning_team
        if fill_competition_stage_table:
            stage_table_df.loc[row.Index, "predicted_home_goals"] = home_goals
            stage_table_df.loc[row.Index, "predicted_away_goals"] = away_goals
            stage_table_df.loc[row.Index, "corners"] = home_corners + away_corners
            stage_table_df.loc[row.Index, "yellow_cards"] = home_yellow + away_yellow
            stage_table_df.loc[row.Index, "red_cards"] = home_red + away_red


        # =========================
        # STORE RESULT
        # =========================
        result_record = {
            "group": group,
            "predicted_home_team": home,
            "predicted_away_team": away,
            "home_goals": home_goals,
            "away_goals": away_goals,
            "home_yellow": home_yellow,
            "away_yellow": away_yellow,
            "home_red": home_red,
            "away_red": away_red,
            "home_corners": home_corners,
            "away_corners": away_corners,
        }
        results.append(result_record)

        # =========================
        # UPDATE STATS
        # =========================

        # goals for / against
        stats[home]["gf"] += home_goals
        stats[home]["ga"] += away_goals

        stats[away]["gf"] += away_goals
        stats[away]["ga"] += home_goals

        # yellow cards
        stats[home]["yc"] += home_yellow
        stats[away]["yc"] += away_yellow

        # red cards
        stats[home]["rc"] += home_red
        stats[away]["rc"] += away_red

        # corners
        stats[home]["corners"] += home_corners
        stats[away]["corners"] += away_corners

        # result
        if result_str == "W":
            stats[home]["pts"] += 3
            stats[home]["w"] += 1
            stats[away]["l"] += 1
            if fill_competition_stage_table:
                stage_table_df.loc[row.Index, "winning_team"] = "home"
        elif result_str == "L":
            stats[away]["pts"] += 3
            stats[away]["w"] += 1
            stats[home]["l"] += 1
            if fill_competition_stage_table:
                stage_table_df.loc[row.Index, "winning_team"] = "away"
        else:
            stats[home]["pts"] += 1
            stats[away]["pts"] += 1
            stats[home]["d"] += 1
            stats[away]["d"] += 1
            if fill_competition_stage_table:
                stage_table_df.loc[row.Index, "winning_team"] = "draw"

        # =========================
        # GOAL DIFFERENCE
        # =========================
        stats[home]["gd"] = stats[home]["gf"] - stats[home]["ga"]
        stats[away]["gd"] = stats[away]["gf"] - stats[away]["ga"]

        # =========================
        # SORT TEAMS - FIFA tiebreakers: Points → Goal Difference → Goals Scored
        # =========================
        ranking = sorted(
            group_teams,
            key=lambda x: (
                stats[x]["pts"],
                stats[x]["gd"],
                stats[x]["gf"]
            ),
            reverse=True # sort in descending order
        )

        # =========================
        # PRINT STATS
        # =========================
        if verbose:
            print(f"\n{'═' * 60}")
            print(f"  GROUP {group}")
            print(f"{'═' * 60}\n")

            print(f"  {'#':<3} {'Team':<16} {'Pts':<4} {'W':<4} {'D':<4} {'L':<4} {'GF':<4} {'GA':<4} {'GD':<4} {'Corners':<4} {'YC':<4} {'RC':<4}")
            print(f"{'-' * 60}")

            for i, t in enumerate(ranking):
                s = stats[t]
                tag =" ✅" if i <= 2 else (" ⏳" if i == 3 else " ❌")

                print(f"{i+1:<3} {t:<16} {s['pts']:>4} {s['w']:>4} "
                f"{s['d']:>4} {s['l']:>4} {s['gf']:>4} {s['ga']:>4} "
                f"{s['gd']:>4} {s['corners']:>4} {s['yc']:>4} "
                f"{s['rc']:>4} {tag}")

            print(f"\n{'═' * 60}")
            print(f"\n Match Results:")
            for i, row in stage_table_df.iterrows():
                print(f"{row.home_team} {row.home_goals} - {row.away_goals} {row.away_team}")
            
    return {
        "group_stats": group_stats,
        "results": results,
        "stage_table_df": stage_table_df
    }

In [303]:
import copy
def get_round_of_32(group_stats, verbose = True):
    """
    Top 2 from each group (24) + 8 best 3rd-placed teams = 32.
    Tiebreakers for 3rd place: Pts → GD → GF.
    """

    # =========================
    # PREP DATA
    # =========================
    group_stats = copy.deepcopy(group_stats)

    qualifiers = []
    third_pool = []

    # =========================
    # PROCESS EACH GROUP
    # =========================
    for gname, teams_stats in group_stats.items():

        # FINAL RANKING
        ranking = sorted(
            teams_stats.keys(),
            key=lambda x: (
                teams_stats[x]["pts"],
                teams_stats[x]["gd"],
                teams_stats[x]["gf"],
                -teams_stats[x]["yc"], # - to sort in descending order
                -teams_stats[x]["rc"]
            ),
            reverse=True
        )

        # =========================
        # TOP 2 QUALIFY
        # =========================
        # top 1
        qualifiers.append(
            {
                "team": ranking[0],
                "group": gname,
                "pos":  1
            }
        )

        # top 2
        qualifiers.append(
            {
                "team": ranking[1],
                "group": gname,
                "pos":  2
            }
        )

        # =========================
        # 3RD PLACED TEAMS
        # =========================
        if len(ranking) >= 3:
            t = ranking[2]
            s = teams_stats[t]

            third_pool.append({
                "team": t,
                "group": gname,
                "pts": s["pts"],
                "gd": s["gd"],
                "gf": s["gf"],
                "yc": s["yc"],
                "rc": s["rc"],
                "pos": 3
            })

    # =========================
    # SORT 3RD POOL
    # =========================
    third_pool.sort(
        key=lambda x: (
            x["pts"],
            x["gd"],
            x["gf"],
            -x["yc"],
            -x["rc"]
        ),
        reverse=True
    )

    # =========================
    # GET BEST 8 3RD PLACED TEAMS
    # =========================
    best_third = third_pool[:8]
    r32 = qualifiers + [
            {
                "team": t["team"],
                "group": t["group"],
                "pos": t["pos"]
            }
            for t in best_third
        ]

    if verbose:
        eliminated = third_pool[8:]
        print(f"\n{'═' * 60}")
        print(f"  ROUND OF 32 - QUALIFICATION SUMMARY")
        print(f"{'═' * 60}\n")
        print(f"\n  🟢 Best 3rd-Place Teams (advance to R32):")
        for i, (team, group, pts, gd, gf, _, _, _) in enumerate(best_third):
            print(f"  {i+1:<2}. {team:<16} {group} {pts:>3} {gd:>3} {gf:>3}")
        print(f"\n  🔴 Eliminated Teams (remain in 3rd Pool):")
        for i, (team, group, pts, gd, gf, _, _) in enumerate(eliminated):
            print(f"  {i+1:<2}. {team:<16} {group} {pts:>3} {gd:>3} {gf:>3}")
        print(f"\n{'═' * 60}")
        print(f"\n  📊 Total Qualifiers: {len(qualifiers)}")

    return {
        "round_name": "Round of 32",
        "qualifiers": qualifiers,
        "third_pool": third_pool,
        "r32": r32
    }

In [304]:
def knockout_map(previous_round_results):
    """
    This is used to map knockout brackets from quarter final -> final
    """
    winner_map = {}
    loser_map = {}

    for match in previous_round_results["results"]:

        match_id = match["match_id"]

        winner = match["match_winner"]

        # ---------------------------------------------
        # Determine loser
        # ---------------------------------------------
        loser = (
            match["predicted_away_team"] if winner == match["predicted_home_team"]
            else match["predicted_home_team"]
        )

        winner_map[match_id] = winner
        loser_map[match_id] = loser

    return winner_map, loser_map
    

In [305]:
def fill_knockout_table(knockout_df, round_name, qualifiers, previous_round_results=None):
    """
    Populate knockout slots with actual teams.

    Parameters
    -------------------------------
    knockout_df : DataFrame
        Full knockout bracket dataframe.

    round_name : str
        Current round name.

    qualifiers : list[dict]
        Qualifiers of group stage

    previous_round_results : dict
        Results dict from previous knockout round.
        Must contain in "results" key:
            - match_id
            - match_winner
            - predicted_home_team
            - predicted_away_team
    """

    # =====================================================
    # FILTER ROUND
    # =====================================================
    knockout_df = knockout_df[
            knockout_df["round"] == round_name
        ].copy()

    # =====================================================
    # ROUND OF 32
    # =====================================================

    if round_name == "Round of 32":
        # =========================
        # BUILD LOOKUPS
        # =========================
        group_winners = {}
        group_runners = {}
        group_thirds = []

        for qualifier in qualifiers:
            if qualifier["pos"] == 1:
                group_winners[qualifier["group"]] = qualifier["team"]
            elif qualifier["pos"] == 2:
                group_runners[qualifier["group"]] = qualifier["team"]
            elif qualifier["pos"] == 3:
                group_thirds.append(qualifier["team"])

        third_index = 0

        # =========================
        # SLOT RESOLVER
        # =========================
        def resolve(slot):
            nonlocal third_index

            # Winner Group X
            if "Winner Group" in slot:
                group = slot.split()[-1]
                return group_winners.get(group)

            # Runner-up Group X
            if "Runner-up Group" in slot:
                group = slot.split()[-1]
                return group_runners.get(group)

            # Best 3rd
            if "Best 3rd" in slot:
                if third_index >= len(group_thirds):
                    return None
                team = group_thirds[third_index]
                third_index += 1
                return team

            return None  # strict fallback (prevents silent bad data)

    # =====================================================
    # ALL OTHER ROUNDS (R16, R8, R4, Final, Third place)
    # =====================================================
    else:
        winner_map, loser_map = knockout_map(previous_round_results)
        
        def resolve(slot):
            
            if slot is None:
                return None

            if not isinstance(slot, str):
                return slot

            # ---------------------------------------------
            # Winner Match X
            # ---------------------------------------------
            if "Winner Match" in slot:

                match_id = int(slot.split()[-1])

                return winner_map.get(match_id)

            # ---------------------------------------------
            # Loser Match X
            # ---------------------------------------------
            if "Loser Match" in slot:

                match_id = int(slot.split()[-1])

                return loser_map.get(match_id)
        
            return slot

    # =====================================================
    # FILL TEAMS
    # =====================================================
    knockout_df = knockout_df.copy()

    knockout_df["predicted_home_team"] = knockout_df["slot_home"].apply(resolve)
    knockout_df["predicted_away_team"] = knockout_df["slot_away"].apply(resolve) 
        
    return knockout_df

In [306]:
def knockout_simulate(round_name, qualifiers_data, knockout_df, simulate_match_fn, models, elo_ratings, team_hist, feature_engine, fill_knockout_table_bool = False, previous_round_results = None, verbose = True):
    """
    Simulate a knockout round (Round of 32, 16, QF, SF, Final).
    """
    import pandas as pd
    if verbose:
        print(f"\n{'═' * 60}")
        print(f"  {round_name}")
        print(f"{'═' * 60}\n")
        print(f"  {'#':<4} {'Home':<16} {'':>2}{'Score':>7}{'':>2} "
              f"{'Away':<16}  {'Winner'}")
        print(f"  {'─' * 60}")
    knockout_df = knockout_df.copy()

    if fill_knockout_table_bool:
        if qualifiers_data["round_name"] == "Round of 32":
            knockout_df = fill_knockout_table(
                knockout_df,
                round_name,
                qualifiers_data["r32"],
                None
            )
        else:
            knockout_df = fill_knockout_table(
                knockout_df,
                round_name,
                qualifiers_data["r32"], # backup
                previous_round_results
            )

    knockout_df["date"] = pd.to_datetime(knockout_df["date_utc"], utc=True)

    # =====================================================
    # FILTER MATCHES
    # =====================================================
    matches = knockout_df[knockout_df["round"] == round_name].copy().reset_index(drop=True)

    # =====================================================
    # IF NOT PRE-FILLED → RESOLVE HERE
    # =====================================================
    if not fill_knockout_table_bool: 
        if round_name == "Round of 32":
            matches = fill_knockout_table(
                matches,
                round_name,
                qualifiers_data["r32"],
                None
            )
        else:
            winner_map, loser_map = knockout_map(previous_round_results)

            def resolve(slot):
                if slot is None:
                    return None

                if not isinstance(slot, str):
                    return slot

                # ---------------------------------------------
                # Winner Match X
                # ---------------------------------------------
                if "Winner Match" in slot:

                    match_id = int(slot.split()[-1])

                    return winner_map.get(match_id)

                # ---------------------------------------------
                # Loser Match X
                # ---------------------------------------------
                if "Loser Match" in slot:

                    match_id = int(slot.split()[-1])

                    return loser_map.get(match_id)
            
                return slot

            # fill matching teams
            matches["predicted_home_team"] = matches["slot_home"].apply(resolve)
            matches["predicted_away_team"] = matches["slot_away"].apply(resolve)
    

    results = []

    # =========================
    # SIMULATE EACH MATCH
    # =========================
    for i, row in matches.iterrows():
        home, away = row["predicted_home_team"], row["predicted_away_team"]
        date = row["date"]

        match_row = pd.DataFrame({
            "home_team": [home],
            "away_team": [away],
            "date": [date]
        })

        result = simulate_match_fn(
            match_row,
            models,
            elo_ratings,
            team_hist,
            feature_engine,
            knockout=True,
            et_total=0.8
        )

        home_goals = result["home_goals"]
        away_goals = result["away_goals"]
        home_yellow = result["home_yellow"]
        away_yellow = result["away_yellow"]
        home_red = result["home_red"]
        away_red = result["away_red"]
        home_corners = result["home_corners"]
        away_corners = result["away_corners"]
        isPenalty = result["penalty"]
        result_str = result["result_str"]

        if fill_knockout_table_bool:
            knockout_df.loc[i, "predicted_home_goals"] = home_goals
            knockout_df.loc[i, "predicted_away_goals"] = away_goals
            knockout_df.loc[i, "corners"] = home_corners + away_corners
            knockout_df.loc[i, "yellow_cards"] = home_yellow + away_yellow
            knockout_df.loc[i, "red_cards"] = home_red + away_red
            knockout_df.loc[i, "penalties"] = isPenalty

        match_winner = None
        match_loser = None
        if result_str == "W":
            match_winner = home
            match_loser = away
            if fill_knockout_table_bool:
                knockout_df.loc[i, "match_winner"] = "home"
        elif result_str == "L":
            match_winner = away
            match_loser = home
            if fill_knockout_table_bool:
                knockout_df.loc[i, "match_winner"] = "away"


        results.append({
            "match_id": row["match_id"],
            "round": round_name,
            "predicted_home_team": home,
            "predicted_away_team": away,
            "home_goals": home_goals,
            "away_goals": away_goals,
            "match_winner": match_winner,
            "match_loser": match_loser
        })

        if verbose:
            print(
                f"{row['match_id']:<4} "
                f"{home:<16} {home_goals}-{away_goals:<5} "
                f"{away:<16} => {match_winner}"
            )

    return {
        "round_name": round_name,
        "results": results,
        "knockout_df": knockout_df
    }

In [307]:
def world_cup_simulation(
    stage_table_df,
    knockout_df,
    simulate_match_fn,
    models,
    elo_ratings,
    team_hist,
    feature_engine,
    fill_competition_stage_table = False,
    verbose = True, 
    seed = None
    ):
    """ Run a complete FIFA World Cup 2026 simulation. """
    import random
    import numpy as np
    if seed is not None:
        random.seed()
        np.random.seed(seed)

    if verbose:
        banner = "  🏆  FIFA WORLD CUP 2026 — FULL SIMULATION  🏆"
        print(f"\n{'═' * 60}")
        print(banner)
        print(f"  Hosted by USA · CANADA · MEXICO")
        print(f"  48 Teams  |  12 Groups  |  R32 → Final")
        print(f"{'═' * 60}")

    # ── Group Stage ──
    # Run simulate_group_stage
    group_stage_return = simulate_group_stage(
        stage_table_df, 
        simulate_match_fn, 
        models, 
        elo_ratings,
        team_hist,
        feature_engine,
        fill_competition_stage_table,
        verbose
        )

    group_stats = group_stage_return["group_stats"]

    # Run get_round_of_32
    get_round_of_32_return = get_round_of_32(
        group_stats,
        verbose
    )


    # Run knockout_simulate
    # ==================================
    # ROUND OF 32
    # ==================================
    r32_return = knockout_simulate(
        "Round of 32",
        get_round_of_32_return,
        knockout_df,
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        fill_knockout_table_bool=True,
        previous_round_results=None,
        verbose=verbose
    )

    # ==================================
    # ROUND OF 16
    # ==================================
    r16_return = knockout_simulate(
        "Round of 16",
        get_round_of_32_return,
        r32_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        fill_knockout_table_bool=True,
        previous_round_results=r32_return,
        verbose=verbose
    )

    # ==================================
    # QUARTER-FINAL
    # ==================================
    qf_return = knockout_simulate(
        "Quarter-final",
        get_round_of_32_return,
        r16_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        fill_knockout_table_bool=True,
        previous_round_results=r16_return,
        verbose=verbose
    )

    # ==================================
    # SEMI-FINAL
    # ==================================
    sf_return = knockout_simulate(
        "Semi-final",
        get_round_of_32_return,
        qf_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        fill_knockout_table_bool=True,
        previous_round_results=qf_return,
        verbose=verbose
    )

    # ==================================
    # FINAL
    # ==================================
    f_return = knockout_simulate(
        "Final",
        get_round_of_32_return,
        sf_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        fill_knockout_table_bool=True,
        previous_round_results=sf_return,
        verbose=verbose
    )

    final_home_team = f_return["results"][0]["predicted_home_team"]
    final_away_team = f_return["results"][0]["predicted_away_team"]
    final_home_score = f_return["results"][0]["home_goals"]
    final_away_score = f_return["results"][0]["away_goals"]
    champion = f_return["results"][0]["match_winner"]
    runner_up = f_return["results"][0]["match_loser"]

    # ==================================
    # THIRD PLACE
    # ==================================
    third_place_return = knockout_simulate(
        "Third-place playoff",
        get_round_of_32_return,
        sf_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        fill_knockout_table_bool=True,
        previous_round_results=sf_return,
        verbose=verbose
    )
    third = third_place_return["results"][0]["match_winner"]


    if verbose:
        print(f"\n{'🏆' * 30}")
        print(f"                      FINAL")
        print(f"{'🏆' * 30}")
        print(f"  {final_home_team:<16} {final_home_score} - {final_away_score} {final_away_team:<16}")
        print(f"\n  🏆  WORLD CHAMPION :  {champion}")
        print(f"  🥈  RUNNER-UP      :  {runner_up}")
        print(f"  🥉  THIRD PLACE    :  {third}")
        print(f"{'🏆' * 30}\n")

    return {"champion": champion, "runner_up": runner_up, "third": third}

In [308]:
import import_ipynb
import preprocessing

team_hist = preprocessing.team_hist
combine_teams_elo_fn = preprocessing.combine_teams_elo
is_home_advantage_fn = preprocessing.is_home_advantage
add_match_features_fn = preprocessing.add_match_features

In [309]:
elo = pd.read_csv('data/elo.csv', on_bad_lines="skip")

In [310]:
import pickle

with open("models/goal_home.pkl", "rb") as f:
    goal_home_best = pickle.load(f)

with open("models/goal_away.pkl", "rb") as f:
    goal_away_best = pickle.load(f)

with open("models/yellow_home.pkl", "rb") as f:
    yellow_home_best = pickle.load(f)

with open("models/yellow_away.pkl", "rb") as f:
    yellow_away_best = pickle.load(f)

with open("models/red_home.pkl", "rb") as f:
    red_home_best = pickle.load(f)

with open("models/red_away.pkl", "rb") as f:
    red_away_best = pickle.load(f)
 
with open("models/corner_home.pkl", "rb") as f:
    corner_home_best = pickle.load(f)

with open("models/corner_away.pkl", "rb") as f:
    corner_away_best = pickle.load(f)

In [ ]:
feature_engine = {
       "combine_elo": combine_teams_elo_fn,
       "home_adv": is_home_advantage_fn,
       "match_features": add_match_features_fn
    }

In [312]:
models = {
    "goal_home": goal_home_best,
    "goal_away": goal_away_best,
    "yellow_home": yellow_home_best,
    "yellow_away": yellow_away_best,
    "red_home": red_home_best,
    "red_away": red_away_best,
    "corner_home": corner_home_best,
    "corner_away": corner_away_best
}


In [313]:
stage_df = pd.read_csv(r"data/group_fixtures.csv")
knockout_df = pd.read_csv(r"data/knockout_slots.csv")

In [314]:
world_cup_simulation(stage_df, knockout_df, simulate_match, models, elo, team_hist, feature_engine, True, True, 42)


════════════════════════════════════════════════════════════
  🏆  FIFA WORLD CUP 2026 — FULL SIMULATION  🏆
  Hosted by USA · CANADA · MEXICO
  48 Teams  |  12 Groups  |  R32 → Final
════════════════════════════════════════════════════════════
reach
  home_team     away_team                      date  home_elo  away_elo  \
0    Mexico  South Africa 2026-06-11 19:00:00+00:00      1858      1524   

   elo_diff  home_advantage  
0       334               1  
reach


C:\Users\Hi\AppData\Local\Temp\ipykernel_3960\3632572644.py:84: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  match_row["home_advantage"] = int(


TypeError: Cannot compare tz-naive and tz-aware timestamps

In [ ]:
# def monte_carlo(elo_ratings, n=1000, seed=42):
#     """
#     Run N full tournament simulations silently and return
#     an aggregated statistics DataFrame.
#     """
#     champions, runners_up, thirds = [], [], []

#     for i in range(n):
#         s = seed + i if seed is not None else None
#         r = world_cup_simulation(verbose=False, seed=s)
#         champions.append(r["champion"])
#         runners_up.append(r["runner_up"])
#         thirds.append(r["third"])

#     c_cnt = pd.Series(champions).value_counts()
#     r_cnt = pd.Series(runners_up).value_counts()
#     t_cnt = pd.Series(thirds).value_counts()


#     rows = []
